In [10]:
import joblib

X_train = joblib.load("X_train.pkl")
X_val = joblib.load("X_val.pkl")
X_test = joblib.load("X_test.pkl")

ytrain = joblib.load("ytrain.pkl")
yval = joblib.load("yval.pkl")
ytest = joblib.load("ytest.pkl")

print("Variables loaded successfully!")

Variables loaded successfully!


In [11]:
import os

print(os.listdir("/content"))

['.config', 'X_val.pkl', 'X_test.pkl', 'essay_scoring_app', 'FAST_FEATURES.pkl', 'yval.pkl', 'X_train.pkl', 'ytest.pkl', 'cleaned_asap.csv', 'ytrain.pkl', 'sample_data']


In [12]:
from pathlib import Path
import json

import joblib
import numpy as np
import sklearn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

MODEL_DIR = Path('essay_scoring_app/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SCORE_MIN, SCORE_MAX = 1, 6

FAST_FEATURES = [
    'word_count',
    'sentence_count',
    'character_count',
    'paragraph_count',
    'avg_word_length',
    'avg_sentence_length',
    'unique_words',
    'type_token_ratio',
    'stopword_ratio',
    'avg_paragraph_len',
    'flesch_reading_ease',
    'flesch_kincaid_grade',
    'gunning_fog_score',
    'dale_chall_readability_score',
    'guiraud_index',
    'long_word_ratio',
    'conjunction_ratio',
]


def evaluate(y_true, y_pred):
    rounded = np.clip(np.round(y_pred), SCORE_MIN, SCORE_MAX).astype(int)
    return {
        'MSE': float(mean_squared_error(y_true, y_pred)),
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'R2': float(r2_score(y_true, y_pred)),
        'Accuracy': float(accuracy_score(y_true, rounded)),
        'QWK': float(cohen_kappa_score(y_true, rounded, weights='quadratic')),
    }


# Drop the 7 prompt one-hot columns.
train_17 = X_train[FAST_FEATURES]
val_17 = X_val[FAST_FEATURES]
test_17 = X_test[FAST_FEATURES]

deploy_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
)
deploy_model.fit(train_17, ytrain)

val_metrics = evaluate(yval, deploy_model.predict(val_17))
test_metrics = evaluate(ytest, deploy_model.predict(test_17))

print('Gradient Boosting, 17 prompt-independent features')
print('  Validation:', {k: round(v, 4) for k, v in val_metrics.items()})
print('  Test:      ', {k: round(v, 4) for k, v in test_metrics.items()})
print()
print('24-feature reference (from earlier in the notebook):')
print('  Test: MSE 0.4006 | MAE 0.4801 | R2 0.6239 | Acc 0.6243 | QWK 0.7387')
print('Record both sets of numbers in your report.')

Gradient Boosting, 17 prompt-independent features
  Validation: {'MSE': 0.3984, 'MAE': 0.4807, 'R2': 0.6232, 'Accuracy': 0.61, 'QWK': 0.7304}
  Test:       {'MSE': 0.4153, 'MAE': 0.487, 'R2': 0.6102, 'Accuracy': 0.6173, 'QWK': 0.7325}

24-feature reference (from earlier in the notebook):
  Test: MSE 0.4006 | MAE 0.4801 | R2 0.6239 | Acc 0.6243 | QWK 0.7387
Record both sets of numbers in your report.


In [13]:
import sys
print(sys.executable)

/usr/bin/python3


In [14]:
medians = train_17.median()

metadata = {
    'model_type': 'GradientBoostingRegressor',
    'model_params': deploy_model.get_params(),
    'feature_columns': FAST_FEATURES,
    'medians': {k: float(v) for k, v in medians.items()},
    'score_min': SCORE_MIN,
    'score_max': SCORE_MAX,
    'n_train': int(len(train_17)),
    'sklearn_version': sklearn.__version__,
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'notes': (
        'Trained on ASAP 2.0 source-based essays, scores 1-6. The 7 prompt '
        'one-hot columns used during notebook experimentation were dropped so '
        'the model can score essays on arbitrary topics.'
    ),
}

joblib.dump(deploy_model, MODEL_DIR / 'gb_model.joblib')

with open(MODEL_DIR / 'model_meta.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Exported:')
print(' ', MODEL_DIR / 'gb_model.joblib')
print(' ', MODEL_DIR / 'model_meta.json')
print()
print('scikit-learn version:', sklearn.__version__)
print('Pin this exact version in requirements.txt.')

Exported:
  essay_scoring_app/models/gb_model.joblib
  essay_scoring_app/models/model_meta.json

scikit-learn version: 1.6.1
Pin this exact version in requirements.txt.


In [15]:
try:
    from google.colab import files
    files.download(str(MODEL_DIR / 'gb_model.joblib'))
    files.download(str(MODEL_DIR / 'model_meta.json'))
except ImportError:
    print('Not running in Colab — the files are already on disk at:')
    print(' ', MODEL_DIR.resolve())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
# Optional check — requires feature_extractor.py to be importable.
import pandas as pd

try:
    from feature_extractor import extract_features

    idx = X_train.index[0]
    app_side = extract_features(df.loc[idx, 'full_text'])

    comparison = pd.DataFrame({
        'training': X_train.loc[idx, FAST_FEATURES],
        'app': pd.Series(app_side),
    })
    comparison['abs_diff'] = (comparison['training'] - comparison['app']).abs()
    display(comparison.round(4))
except ImportError:
    print('Upload feature_extractor.py to this environment to run the check.')

Upload feature_extractor.py to this environment to run the check.
